In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/app_train_raw.csv')
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loaded: 307,511 rows, 122 columns


In [2]:
# Drop columns with more than 40% missing values
threshold = 0.4
missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

print(f"Columns to drop (>40% missing): {len(cols_to_drop)}")
print(cols_to_drop[:10], "...")

df = df.drop(columns=cols_to_drop)
print(f"\nAfter dropping: {df.shape[1]} columns remaining")

Columns to drop (>40% missing): 49
['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG'] ...

After dropping: 73 columns remaining


## Notebook 2 — Data Cleaning & Feature Engineering

The raw dataset came in with 122 features and a lot of noise. My goal in
this notebook was to get it into a shape that a machine learning model
can actually learn from — dropping irrelevant columns, fixing data types,
engineering meaningful new features, and encoding categoricals properly.

## Dropping High-Missing Columns

I set a threshold of 40% missing values. Any column missing more than
40% of its data is more likely to introduce noise than signal, so I
dropped them entirely. This removed 49 columns, mostly property-related
features like apartment area measurements and building characteristics.

These columns were missing because many applicants simply don't own
property — so the absence itself might be informative, but the raw
columns are too sparse to be useful directly.

In [3]:
# Convert negative days to positive meaningful values
df['AGE_YEARS'] = abs(df['DAYS_BIRTH']) / 365
df['EMPLOYMENT_YEARS'] = abs(df['DAYS_EMPLOYED'].clip(upper=0)) / 365

# Flag anomaly in DAYS_EMPLOYED (365243 means unemployed/pensioner)
df['IS_UNEMPLOYED'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)

# Drop original columns
df = df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION'])

print("Age statistics:")
print(df['AGE_YEARS'].describe().round(2))
print("\nEmployment years statistics:")
print(df['EMPLOYMENT_YEARS'].describe().round(2))

Age statistics:
count    307511.00
mean         43.94
std          11.96
min          20.52
25%          34.01
50%          43.15
75%          53.92
max          69.12
Name: AGE_YEARS, dtype: float64

Employment years statistics:
count    307511.00
mean          5.36
std           6.32
min           0.00
25%           0.79
50%           3.32
75%           7.56
max          49.07
Name: EMPLOYMENT_YEARS, dtype: float64


## Converting Days to Meaningful Features

The original dataset stored age and employment duration as negative numbers
counting backwards from the application date — for example, -15,000 days
means the person was born 15,000 days before applying.

I converted these into actual years which are much more interpretable.
I also flagged a specific anomaly — DAYS_EMPLOYED had a value of 365,243
for pensioners and unemployed applicants, which is clearly a placeholder
rather than a real employment duration. I created a separate IS_UNEMPLOYED
flag to capture this information properly instead of letting it corrupt
the employment years calculation.

In [4]:
# Remove anomaly in gender
df = df[df['CODE_GENDER'] != 'XNA'].copy()

# Label encode binary categoricals
binary_cols = ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
               'NAME_CONTRACT_TYPE', 'EMERGENCYSTATE_MODE']

le = LabelEncoder()
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')
        df[col] = le.fit_transform(df[col].astype(str))

# One hot encode remaining categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Remaining categorical columns: {cat_cols}")
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"\nAfter encoding: {df.shape[1]} columns")

Remaining categorical columns: ['NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']

After encoding: 172 columns


## Encoding Categorical Variables

Machine learning models can't directly process text categories, so I
encoded them as numbers. For binary categories like gender, car ownership,
and realty ownership I used Label Encoding which converts them to 0 and 1.

For the remaining 8 multi-category columns like income type, education
type, and organization type I used One-Hot Encoding which creates a
separate binary column for each category. This expanded the dataset
from 73 to 172 columns but ensures no artificial ordering is implied
between categories.

I also removed 4 rows where gender was listed as XNA since that's
clearly a data entry error rather than a valid category.

In [5]:
# Fill remaining missing values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"Final shape: {df.shape}")

# Save clean data
df.to_csv('../data/credit_clean.csv', index=False)
print("Saved to data/credit_clean.csv!")

Missing values remaining: 0
Final shape: (307507, 172)
Saved to data/credit_clean.csv!


## Imputing Missing Values and Saving

For the remaining missing values in numeric columns I used median
imputation — filling each missing value with the median of that column.
Median is more robust than mean for financial data because extreme
outliers in income and loan amounts would skew the mean significantly.

After all cleaning steps the dataset has zero missing values and is
ready for modeling. The final shape of 172 features gives the model
plenty of signals to learn from while keeping the most informative
columns intact.